In [ ]:
import os
import torch
from torch_geometric.data import InMemoryDataset, HeteroData

def initialize_graph():
    data = HeteroData()
    return data


In [ ]:
# Variables: [numVariableNodes, feature_dim]
# Assuming encoded_variable_node_indices is already a tensor
x_vars = torch.tensor(estimate_row, dtype=torch.float).view(-1, 1)
data['variable'].x = torch.cat([x_vars, encoded_variable_node_indices], dim=-1)

# Factors: [numMeasurements, 3]
meas = torch.tensor(measurement_rows[iGraph], dtype=torch.float)
var = (torch.log10(torch.tensor(variance_rows[iGraph], dtype=torch.float)) / 10.0)
cov = torch.tensor(covariance_row, dtype=torch.float) * 10.0

data['factor'].x = torch.stack([meas, var, cov], dim=-1)

In [ ]:
# Assume jacobian_matrix is a [numMeasurements x numVariableNodes] tensor
# This single line replaces your entire add_graph_edges loop
edge_index_fv = torch.nonzero(torch.abs(jacobian_matrix) > 0.0001).t()

# Add to HeteroData (Factor to Variable)
data['factor', 'to', 'variable'].edge_index = edge_index_fv

# Add Reverse Edges (Variable to Factor) to simulate undirected behavior
data['variable', 'to', 'factor'].edge_index = edge_index_fv.flip(0)

In [ ]:
import os
import torch
from torch_geometric.data import InMemoryDataset, HeteroData

class PowerGridDataset(InMemoryDataset):
    def __init__(self, root, dataset_type, transform=None, pre_transform=None):
        # dataset_type should be "train", "validation", or "test"
        self.dataset_type = dataset_type 
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_dir(self):
        # Directory where your raw CSV files are stored
        return os.path.join(self.root, 'data_from_wls_se_solver')

    @property
    def processed_file_names(self):
        # Saves separate files for train/test/val
        return [f'{self.dataset_type}_data.pt']

    def process(self):
        data_list = []
        
        print(f"Loading {self.dataset_type} CSVs into memory...")
        # Assuming read_from_wls_se_files and encode_variable_node_indices are accessible here
        numGraphs, numVariableNodes, numMeasurements, estimate_rows, jacobian_rows, measurement_rows, variance_rows, covariance_rows = read_from_wls_se_files(self.dataset_type, self.raw_dir)
        
        # Compute encodings once for the whole dataset
        encoded_indices = torch.tensor(encode_variable_node_indices(numVariableNodes), dtype=torch.float)

        print(f"Vectorizing {numGraphs} graphs into HeteroData objects...")
        for iGraph in range(numGraphs):
            if iGraph % 100 == 0 and iGraph > 0:
                print(f"Processed {iGraph}/{numGraphs} graphs...")

            data = HeteroData()
            
            # ---------------------------------------------------------
            # 1. VARIABLE NODES (Buses/States)
            # ---------------------------------------------------------
            est_row = [float(x) for x in estimate_rows[iGraph]]
            x_vars = torch.tensor(est_row, dtype=torch.float).view(-1, 1)
            data['variable'].x = torch.cat([x_vars, encoded_indices], dim=-1)
            
            # ---------------------------------------------------------
            # 2. FACTOR NODES (Measurements)
            # ---------------------------------------------------------
            meas = torch.tensor([float(x) for x in measurement_rows[iGraph]], dtype=torch.float)
            var = torch.log10(torch.tensor([float(x) for x in variance_rows[iGraph]], dtype=torch.float)) / 10.0
            
            # Vectorized version of your old `for x in covariance_row_temp: append(x)` doubling loop
            cov_temp = torch.tensor([float(x) for x in covariance_rows[iGraph]], dtype=torch.float)
            cov = torch.repeat_interleave(cov_temp, 2) * 10.0
            
            data['factor'].x = torch.stack([meas, var, cov], dim=-1)
            
            # ---------------------------------------------------------
            # 3. BIPARTITE EDGES (Factor <-> Variable)
            # ---------------------------------------------------------
            # Your Jacobian list was 1D (flattened across all graphs).
            # We slice out the specific M rows for the current graph.
            start_idx = iGraph * numMeasurements
            end_idx = start_idx + numMeasurements
            jacob_block = torch.tensor(jacobian_rows[start_idx:end_idx], dtype=torch.float)
            
            # Find non-zero indices (row, col) equivalent to your > 0.0001 check
            edge_index_fv = torch.nonzero(torch.abs(jacob_block) > 0.0001).t().contiguous()
            
            data['factor', 'to', 'variable'].edge_index = edge_index_fv
            data['variable', 'to', 'factor'].edge_index = edge_index_fv.flip(0)
            
            # ---------------------------------------------------------
            # 4. AUGMENTED EDGES (Variable <-> Variable Second Neighbors)
            # ---------------------------------------------------------
            # Replaces the slow nx.single_source_shortest_path_length loop.
            # Mathematically, nodes share a measurement if J^T * J > 0
            binary_jacob = (torch.abs(jacob_block) > 0.0001).float()
            adj_vv = torch.matmul(binary_jacob.t(), binary_jacob)
            
            edge_index_vv = torch.nonzero(adj_vv > 0).t().contiguous()
            data['variable', 'to', 'variable'].edge_index = edge_index_vv

            # Append completed graph to the dataset list
            data_list.append(data)
            
        print("Saving compiled dataset to disk...")
        # PyG collates the list of graphs into a highly efficient massive sparse matrix
        torch.save(self.collate(data_list), self.processed_paths[0])